# ARIMAX Training on `micro_mobility_training_data_2025.csv`

This notebook trains station-wise ARIMAX (`SARIMAX` with exogenous variables) using the same 7-day chronological holdout.

Note: ARIMAX is expensive at this scale. Default is limited to top-traffic stations for practical runtime.


## Colab Setup

- This notebook auto-detects Colab and mounts Google Drive.
- Update `COLAB_PROJECT_ROOT` if your Drive path is different.
- ARIMAX is compute-heavy; start with `MAX_STATIONS = 30` in Colab.


In [4]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX


In [5]:
# Paths and config
# Set this to your repo folder under Google Drive (same structure as local project)
COLAB_PROJECT_ROOT = Path('/content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction')

IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = COLAB_PROJECT_ROOT
else:
    # local fallback
    PROJECT_ROOT = Path.cwd().resolve().parents[1]

DATA_PATH = PROJECT_ROOT / 'data' / 'proceed' / 'micro_mobility_training_data_2025.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'model_training' / 'arimax' / 'v1'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_DAYS = 7
MAX_STATIONS = 30   # set to None to run all stations (much slower)
MIN_TRAIN_POINTS = 24 * 21  # require at least 3 weeks for a station
ORDER = (1, 0, 1)
SEASONAL_ORDER = (1, 0, 1, 24)

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_PATH:', DATA_PATH)
print('ARTIFACT_DIR:', ARTIFACT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction
DATA_PATH: /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/data/proceed/micro_mobility_training_data_2025.csv
ARTIFACT_DIR: /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/arimax/v1


In [6]:
# Load and prepare base frame
cols = [
    'station', 'date', 'hour', 'net_demand',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h'
]

df = pd.read_csv(DATA_PATH, usecols=cols)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'station', 'net_demand']).copy()
df['datetime_hour'] = df['date'] + pd.to_timedelta(df['hour'].astype(int), unit='h')

cutoff_date = df['date'].max().normalize() - pd.Timedelta(days=HOLDOUT_DAYS)
print('Rows total:', len(df))
print('Cutoff date:', cutoff_date.date())


Rows total: 4266120
Cutoff date: 2025-12-24


In [7]:
# Select stations by activity (top departures/rows)
station_counts = df['station'].value_counts()
if MAX_STATIONS is None:
    stations = station_counts.index.tolist()
else:
    stations = station_counts.head(MAX_STATIONS).index.tolist()

print('Stations selected:', len(stations))
print(station_counts.head(10))


Stations selected: 30
station
York St & Marin Blvd             8760
1 Ave & E 38 St                  8760
1 Ave & E 6 St                   8760
1 Pl & Clinton St                8760
10 Ave & W 14 St                 8760
10 St & 7 Ave                    8760
11 Ave & W 27 St                 8760
Water St & Fletcher St           8760
Washington St & Laight St        8760
Washington St & Gansevoort St    8760
Name: count, dtype: int64


In [9]:
exog_cols = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h']

import time

all_preds = []
station_metrics = []

warnings.filterwarnings('ignore')

global_t0 = time.perf_counter()

for i, st in enumerate(stations, start=1):
    station_t0 = time.perf_counter()

    sdf = df.loc[df['station'] == st].sort_values('datetime_hour').copy()

    train_mask = sdf['date'] <= cutoff_date
    test_mask = sdf['date'] > cutoff_date

    y_train = sdf.loc[train_mask, 'net_demand'].astype('float64')
    y_test = sdf.loc[test_mask, 'net_demand'].astype('float64')
    X_train = sdf.loc[train_mask, exog_cols].astype('float64')
    X_test = sdf.loc[test_mask, exog_cols].astype('float64')

    if len(y_train) < MIN_TRAIN_POINTS or len(y_test) == 0:
        continue

    try:
        model = SARIMAX(
            endog=y_train,
            exog=X_train,
            order=ORDER,
            seasonal_order=SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit_res = model.fit(disp=False)
        pred = fit_res.forecast(steps=len(y_test), exog=X_test)

        st_rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        st_mae = float(mean_absolute_error(y_test, pred))
        st_sec = time.perf_counter() - station_t0

        station_metrics.append({
            'station': st,
            'train_rows': int(len(y_train)),
            'test_rows': int(len(y_test)),
            'mae': st_mae,
            'rmse': st_rmse,
            'train_seconds': st_sec,
        })

        pred_df = pd.DataFrame({
            'station': st,
            'date': sdf.loc[test_mask, 'date'].values,
            'hour': sdf.loc[test_mask, 'hour'].astype(int).values,
            'y_true': y_test.values,
            'y_pred': np.asarray(pred),
        })
        all_preds.append(pred_df)

        print(f'[{i}/{len(stations)}] OK: {st} | test={len(y_test)} | MAE={st_mae:.4f} | RMSE={st_rmse:.4f} | time={st_sec:.2f}s')
    except Exception as e:
        print(f'[{i}/{len(stations)}] FAIL: {st} | {e}')

global_train_seconds = time.perf_counter() - global_t0
print(f'Total ARIMAX loop time: {global_train_seconds:.2f} sec ({global_train_seconds/60:.2f} min)')


[1/30] OK: York St & Marin Blvd | test=168 | MAE=0.0000 | RMSE=0.0000 | time=56.57s
[2/30] OK: 1 Ave & E 38 St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=62.37s
[3/30] OK: 1 Ave & E 6 St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=89.44s
[4/30] OK: 1 Pl & Clinton St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=78.88s
[5/30] OK: 10 Ave & W 14 St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=38.35s
[6/30] OK: 10 St & 7 Ave | test=168 | MAE=0.0000 | RMSE=0.0000 | time=42.07s
[7/30] OK: 11 Ave & W 27 St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=76.54s
[8/30] OK: Water St & Fletcher St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=41.48s
[9/30] OK: Washington St & Laight St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=41.69s
[10/30] OK: Washington St & Gansevoort St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=41.78s
[11/30] OK: Washington St | test=168 | MAE=0.0000 | RMSE=0.0000 | time=42.94s
[12/30] OK: Washington Pl & 6 Ave | test=168 | MAE=0.0000 | RMSE=0.0000 | time=62.46s
[13/30] 

In [10]:
# Aggregate metrics
if not all_preds:
    raise RuntimeError('No station was successfully trained. Try increasing MAX_STATIONS or lowering MIN_TRAIN_POINTS.')

pred_all = pd.concat(all_preds, ignore_index=True)

mae = float(mean_absolute_error(pred_all['y_true'], pred_all['y_pred']))
rmse = float(np.sqrt(mean_squared_error(pred_all['y_true'], pred_all['y_pred'])))

hourly_mae = pred_all.groupby('hour').apply(
    lambda x: float(np.mean(np.abs(x['y_true'] - x['y_pred'])))
).reset_index(name='mae')

station_metrics_df = pd.DataFrame(station_metrics).sort_values('mae')

print(f'Global RMSE: {rmse:.4f}')
print(f'Global MAE : {mae:.4f}')
print('Stations successfully trained:', len(station_metrics_df))


Global RMSE: 4.0177
Global MAE : 0.2277
Stations successfully trained: 30


In [11]:
# Save artifacts
pred_path = ARTIFACT_DIR / 'predictions.csv'
station_metrics_path = ARTIFACT_DIR / 'station_metrics.csv'
hourly_mae_path = ARTIFACT_DIR / 'hourly_mae.csv'
metrics_path = ARTIFACT_DIR / 'metrics.json'

pred_all.to_csv(pred_path, index=False)
station_metrics_df.to_csv(station_metrics_path, index=False)
hourly_mae.to_csv(hourly_mae_path, index=False)

metrics = {
    'data_path': str(DATA_PATH),
    'model': 'SARIMAX (ARIMAX)',
    'holdout_days': HOLDOUT_DAYS,
    'order': ORDER,
    'seasonal_order': SEASONAL_ORDER,
    'max_stations': MAX_STATIONS,
    'min_train_points': MIN_TRAIN_POINTS,
    'stations_trained': int(len(station_metrics_df)),
    'mae': mae,
    'rmse': rmse,
}

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved:')
print('-', pred_path)
print('-', station_metrics_path)
print('-', hourly_mae_path)
print('-', metrics_path)


Saved:
- /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/arimax/v1/predictions.csv
- /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/arimax/v1/station_metrics.csv
- /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/arimax/v1/hourly_mae.csv
- /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/arimax/v1/metrics.json
